In [1]:
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [6]:
import argparse
import operator
import os
import re
from collections import defaultdict
from itertools import compress
try:
    from functools import reduce
except ImportError:  # python < 2
    pass

import tqdm

import numpy

import h5py

from gwdatafind.utils import filename_metadata

from ligo.segments import segmentlist
from ligo.segments.utils import fromsegwizard

from pycbc import __version__
from pycbc.inject import InjectionSet
from pycbc.io import FieldArray

__author__ = "Cameron Mills <cameron.mills@aei.mpg.de>"

SPLIT_FILENAME = re.compile(r"_(SPLIT\d+)")

NETWORK_IFO_EVENT_ID_REGEX = re.compile(
    r"\Anetwork/(?P<ifo>[A-Z]1)_event_id\Z",
)
EVENT_ID_REGEX = re.compile(r"event_id\Z")

TQDM_BAR_FORMAT = ("{desc}: |{bar}| "
                   "{n_fmt}/{total_fmt} {unit} ({percentage:3.0f}%) "
                   "[{elapsed} | ETA {remaining}]{postfix}")
TQDM_KW = {
    "ascii": " -=#",
    "bar_format": TQDM_BAR_FORMAT,
    "smoothing": 0.05,
}


# -- utilities ----------------------------------

def find_split_files(filelist, split):
    for fn in filelist:
        if (split is None and 'SPLIT' not in fn) or split in fn:
            yield fn


def read_segment_files(segfiles):
    def _read(name):
        with open(name, "r") as f:
            return fromsegwizard(f)

    return segmentlist(reduce(
        operator.or_,
        map(_read, segfiles),
        segmentlist()))


def read_hdf5_triggers(inputfiles, verbose=False):
    """Merge several HDF5 files into a single file

    Parameters
    ----------
    inputfiles : `list` of `str`
        the paths of the input HDF5 files to merge

    outputfile : `str`
        the path of the output HDF5 file to write
    """
    datasets = {}

    def _scan_dataset(name, obj):
        if not name.startswith("network") or not isinstance(obj, h5py.Dataset):
            return
        if NETWORK_IFO_EVENT_ID_REGEX.match(name):
            return
        shape = obj.shape
        dtype = obj.dtype
        try:
            shape = numpy.sum(datasets[name][0] + shape, keepdims=True)
        except KeyError:
            pass
        else:
            assert dtype == datasets[name][1], (
                "Cannot merge {0}/{1}, does not match dtype".format(
                    obj.file.filename, name,
                ))
        datasets[name] = (shape, dtype)

    # get list of datasets
    datasets = {}
    for filename in inputfiles:
        with h5py.File(filename, 'r') as h5f:
            h5f.visititems(_scan_dataset)

    position = defaultdict(int)

    out = {}

    # create datasets
    for dset, (shape, dtype) in datasets.items():
        out[dset[8:]] = numpy.empty(shape, dtype=dtype)

    # copy dataset contents
    for filename in inputfiles:
        with h5py.File(filename, 'r') as h5in:
            for dset in datasets:
                data = h5in[dset][:]
                size = data.shape[0]
                pos = position[dset]
                if EVENT_ID_REGEX.search(dset):
                    out[dset[8:]][pos:pos+size] = data + pos
                else:
                    out[dset[8:]][pos:pos+size] = data
                position[dset] += size

    return out


In [21]:
class fw(object):
    def __init__(self, name):
        self.f = h5py.File(name, 'w')

    def __setitem__(self, name, data):
        col = self.prefix + '/' + name
        self.f.create_dataset(col, data=data,
                                compression='gzip',
                                compression_opts=9,
                                shuffle=True)
    def __getitem__(self, col):
        return self.f[col]

In [20]:
f.f.close()

In [22]:
f = fw("deleteme.hdf")

In [23]:
f.prefix = "H1"

In [24]:
f["end_time"] = [1,2,3]

In [29]:
f["H1/end_time"]

<HDF5 dataset "end_time": shape (3,), type "<i8">

In [30]:
with h5py.File(trigfiles[0], "r") as f:
    

{'event_id': array([    0,     1,     2, ..., 13313, 13314, 13315]),
 'nifo': array([3, 3, 3, ..., 3, 3, 3]),
 'snr_2_filter': array([7.2439756, 7.045177 , 7.221258 , ..., 8.1222725, 8.624431 ,
        8.475271 ], dtype=float32),
 'snr_2_filter_rss': array([7.6360493, 7.47989  , 8.373582 , ..., 8.736949 , 8.893397 ,
        8.709433 ], dtype=float32),
 'template_id': array([ 0,  0,  0, ..., 65, 65, 65]),
 'timeslide_id': array([0, 0, 0, ..., 0, 0, 0])}

In [63]:
f = h5py.File(trigfiles[0], "r")

In [85]:
f["L1"].keys()

<KeysViewHDF5 ['coa_phase_dom', 'coa_phase_sub', 'end_time', 'event_id', 'search', 'sigmasq_dom', 'sigmasq_sub', 'snr_dominant_l', 'snr_subdominant_l', 'template_duration', 'template_hash', 'time_index']>

In [87]:
f["L1"]["end_time"]

<HDF5 dataset "end_time": shape (30162,), type "<f8">

In [65]:
array(f["network"]["H1_event_id"])

array([    0,     1,     2, ..., 30159, 30160, 30161])

In [66]:
array(f["network"]["L1_event_id"])

array([    0,     1,     2, ..., 30159, 30160, 30161])

In [67]:
array(f["network"]["event_id"])

array([    0,     1,     2, ..., 30159, 30160, 30161])

In [92]:
len(unique(array(f["L1"]["time_index"])))

1147

In [99]:
mask = array(f["network"]["template_id"])==0
len(unique(array(f["L1"]["time_index"][mask]))), len(unique(array(f["network"]["event_id"][mask])))

(441, 575)

In [100]:
len(unique(array(f["L1"]["event_id"][mask]))), len(unique(array(f["network"]["event_id"][mask])))

(575, 575)

In [83]:
len(unique((f["V1"]["event_id"]))), len(unique((f["network"]["event_id"])))

(30162, 30162)

In [72]:
array(f["network"]["timeslide_id"])

array([0, 0, 0, ..., 1, 1, 1])

In [53]:
array(f["H1"]["end_time"])

array([1.26773598e+09, 1.26773598e+09, 1.26773605e+09, ...,
       1.26773633e+09, 1.26773633e+09, 1.26773633e+09])

In [41]:
f["H1"].keys()

<KeysViewHDF5 ['coa_phase_dom', 'coa_phase_sub', 'end_time', 'event_id', 'search', 'sigmasq_dom', 'sigmasq_sub', 'snr_dominant_h1', 'snr_subdominant_h1', 'template_duration', 'template_hash', 'time_index']>

In [61]:
trigfiles = ["../inspiral/MINI-TESTING-512-realdata-INJECTIONS-timeslides.hdf"]
triggers = read_hdf5_triggers(trigfiles)

In [62]:
triggers

{'event_id': array([    0,     1,     2, ..., 30159, 30160, 30161]),
 'nifo': array([3, 3, 3, ..., 3, 3, 3]),
 'snr_2_filter': array([7.2439756, 7.045177 , 7.221258 , ..., 7.572826 , 7.587753 ,
        7.5119977], dtype=float32),
 'snr_2_filter_rss': array([7.6360493, 7.47989  , 8.373582 , ..., 7.9244084, 8.107452 ,
        8.212645 ], dtype=float32),
 'template_id': array([ 0,  0,  0, ..., 65, 65, 65]),
 'timeslide_id': array([0, 0, 0, ..., 1, 1, 1])}